In [2]:
# import packages

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import math
from scipy.misc import electrocardiogram
from scipy.signal import find_peaks
from scipy import optimize
from scipy.stats import gaussian_kde
from scipy.optimize import curve_fit
from scipy.interpolate import interpn
from scipy.interpolate import interp1d

import tkinter as tk
import tkinter.filedialog
import tkinter.messagebox
import os
import glob
import time

from ast import literal_eval


### Execute the next cell if you have a jupyter dark theme
Don't if you don't or don't know what this means

https://medium.com/datadriveninvestor/how-can-i-customize-jupyter-notebook-into-dark-mode-7985ce780f38

In [5]:
from jupyterthemes import jtplot
jtplot.style(theme='onedork', context='notebook', ticks=True, grid=False)

#restart the core in case you executed the cell in white theme and cannot see ticks anymore

ModuleNotFoundError: No module named 'jupyterthemes'

In [7]:
  
def gaussian(x, height, center, width):
    return height*np.exp(-(x - center)**2/(2*width**2))

def gaussian_fit_simple_results(data, oversampling = 10, guessed_width = 20):
    
    #########
    #########fit gaussian in left window to find center of band
    #########
    min_x =  np.min(data)
    max_x =  np.max(data)
    if math.isnan(max_x-min_x) or round((max_x-min_x)*oversampling)==0:
        pars, hData_filtered, hData_filtered_x = [], [], []
    else:  
        bins_single = int(round((max_x-min_x)*oversampling))
        # histogram of positions along the x axis
        h_filtered = np.histogram(data)#, bins = bins_single)  
        #histogram height
        hData_filtered = h_filtered[0]
        ###get the center of each bin
        # bin size
        step = (h_filtered[1][1]-h_filtered[1][0])
        hData_filtered_x = h_filtered[1][:len(h_filtered[1])-1]+step/2
        #guess that band is centered on the window
        guess = [np.mean(hData_filtered), np.mean(hData_filtered_x), guessed_width]
        
        try:
            pars, cov = curve_fit(f=gaussian, xdata=hData_filtered_x, ydata=hData_filtered, p0=guess, bounds=(-np.inf, np.inf))
        except:
            pars = []
    return pars, hData_filtered, hData_filtered_x

## Load result of analysis

In [28]:
root = tk.Tk()
loc_filename = tk.filedialog.askopenfilename(parent=root, 
                                             title='Please select the file containing the result of your analysis')
root.withdraw()

all_results = pd.read_csv(loc_filename)

#create a new column to estimate sarcomere length later on
sarcomere_lengths = [0.0]*len(all_results)
A_band_lengths = [0.0]*len(all_results)
all_results['sarcomere_length'] = sarcomere_lengths
#all_results['A_band_length'] = A_band_lengths
all_results.head(3)

,date,year,sample,nanobody,fly_number,selection,probe,dye,zdisk,band_distance,band_positions,band_widths,band_widths_estimate,sarcomere_length
0,20260521,2026,2,Nano39,1,1,P3,Cy3B,1,225.100597,"[14.184141997806138, 16.504766704340764]","[0.5930319285986465, 0.39931695543139206]","[0.17376176000225763, 0.2823315307974702]",0.0
1,20260521,2026,2,Nano39,1,1,P3,Cy3B,2,198.033338,"[44.77227986401048, 46.81386067037349]","[0.31496553420557194, 0.28858115657751926]","[0.25688586242638084, 0.17406719166028323]",0.0
2,20260521,2026,2,Nano39,1,1,P3,Cy3B,3,210.704669,"[73.99219113387139, 76.1644042157343]","[0.35498459019716644, 0.20477575118564512]","[0, 0.1751594847948183]",0.0


## Estimate of sarcomere length

In [31]:
pixel_size = 97

samples = all_results['sample'].unique()
FOVs = all_results.fly_number.unique()
nanobodies = all_results.nanobody.unique()
selections = all_results.selection.unique()
days = all_results.date.unique()
  
for n in days:
    cond5 = all_results['date'] == n
    for i in samples:
        cond1 = all_results['sample'] == i
        for j in FOVs:
            cond2 = all_results['fly_number'] == j
            for k in nanobodies:
                cond3 = all_results['nanobody'] == k
                for l in selections:
                    cond4 = all_results['selection'] == l
                    selection = all_results[cond1 & cond2 & cond3 & cond4 & cond5]
                    positions = selection['band_positions'].apply(literal_eval).to_numpy()
                    zdisks = selection['zdisk']
                    if len(zdisks) > 1:
                        indices = all_results[cond1 & cond2 & cond3 & cond4 & cond5].index
                        sarcomere_length = []
                        for m in range(len(zdisks)-1):
                            #print(m)
                            left = (positions[m][0]+positions[m][1])/2
                            right = (positions[m+1][0]+positions[m+1][1])/2
                            sarcomere_length = pixel_size*(right-left)
                            #Uncomment to calculate A_band length using band positions
                            #a=((positions[m+1][0]-positions[m][1])/2)*pixel_size
                            #print(a)
                            #print(type(sarcomere_length))
                            #all_results.at[indices[m+1], 'A_band_length'] = a
                            all_results.at[indices[m+1], 'sarcomere_length'] = sarcomere_length
                
all_results.head(10)

,date,year,sample,nanobody,fly_number,selection,probe,dye,zdisk,band_distance,band_positions,band_widths,band_widths_estimate,sarcomere_length
0,20260521,2026,2,Nano39,1,1,P3,Cy3B,1,225.100597,"[14.184141997806138, 16.504766704340764]","[0.5930319285986465, 0.39931695543139206]","[0.17376176000225763, 0.2823315307974702]",0.000000
1,20260521,2026,2,Nano39,1,1,P3,Cy3B,2,198.033338,"[44.77227986401048, 46.81386067037349]","[0.31496553420557194, 0.28858115657751926]","[0.25688586242638084, 0.17406719166028323]",2953.515744
2,20260521,2026,2,Nano39,1,1,P3,Cy3B,3,210.704669,"[73.99219113387139, 76.1644042157343]","[0.35498459019716644, 0.20477575118564512]","[0, 0.1751594847948183]",2840.667059
3,20260521,2026,2,Nano39,1,4,P3,Cy3B,1,165.852903,"[13.66924480812322, 15.37906855303467]","[0.1302920096078423, 0.2866688648583605]","[0.13282547308075024, 0.538520955245789]",0.000000
4,20260521,2026,2,Nano39,1,4,P3,Cy3B,2,226.586067,"[43.672045094729796, 46.00798393452478]","[0.23980404924993726, 0.29057569886460516]","[0.1491776350721296, 0.33493790699235065]",2940.638210
5,20260521,2026,2,Nano39,1,7,P3,Cy3B,1,186.961945,"[10.128040566884918, 12.055483301890083]","[0.3217630402825162, 0.31722663471331]","[0.2330946875654544, 0.2993150909079332]",0.000000
6,20260521,2026,2,Nano39,1,7,P3,Cy3B,2,211.713588,"[38.755669566317344, 40.93828387277278]","[0.37441286498148313, 0.2956166946192954]","[0.34473820654929016, 0.3220767804800959]",2789.255834
7,20260521,2026,2,Nano39,1,7,P3,Cy3B,3,173.894327,"[67.40134586748279, 69.19407088747994]","[0.25156429933282676, 0.34686564726890695]","[0.17824079740284973, 0.29492773615011175]",2759.720971
8,20260521,2026,2,Nano39,1,7,P3,Cy3B,4,223.410283,"[94.73456245255029, 97.03776124375979]","[0.2983010721462832, 0.651659769670094]","[0.12228700249895329, 0.10070635747565566]",2676.079987
9,20260521,2026,2,Nano39,1,8,P3,Cy3B,1,228.600450,"[11.58281930229564, 13.939524970638699]","[0.3110954337926715, 0.3088889427774821]","[0.2511486129909719, 0.31892759239705526]",0.000000


In [33]:
# Save results with sarcomere length to avoid extracting sarcomere length each time
filename_with_sarc = r'/Volumes/dataSchnorrer/EqpSchnorrer/Aditya/Experiments/Proj RNAi/Mutant/Nb39/Nano39_ProjRNAi_Mutant_P3_Cy3B_All_results_with_sarc_length_notaveraged.csv'
all_results.to_csv(filename_with_sarc)